<a href="https://colab.research.google.com/github/pichate-k/advance-ML/blob/main/text_mining/instructureAdv_ML_Text_mining_lab_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instructor Solution: Lab 08 Text Mining

## Intended Use

ไฟล์นี้ใช้สำหรับ:

1. สาธิตในชั้นเรียน
2. ตรวจความถูกต้องของ Student Notebook
3. เผยแพร่เฉลยหลังนักศึกษาส่งงาน
4. สร้าง baseline model สำหรับทดลอง prediction

## Expected Runtime

- Google Colab: ประมาณ 2–5 นาที
- Dataset: 800 rows
- Model: TF-IDF + Logistic Regression

## Expected Learning Outcomes

นักศึกษาควรสามารถ:

- อธิบายผลจาก data exploration
- ทำ text cleaning ขั้นพื้นฐานได้
- ใช้ train/test split พร้อม `stratify=y`
- อธิบาย data leakage ได้
- ใช้ Pipeline ที่มี TF-IDF และ Logistic Regression ได้
- อ่าน Precision, Recall, F1-score และ Confusion Matrix ได้
- วิเคราะห์ตัวอย่าง prediction error ได้


In [ ]:
!pip install pandas scikit-learn matplotlib seaborn beautifulsoup4 -q

In [ ]:
import re
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
#Load Dataset
from google.colab import files

uploaded = files.upload()

DATA_FILE = "customer_feedback_sentiment.csv"

df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
display(df.head())

In [ ]:
# Instructor Check Cell
expected_columns = {
    "review_id",
    "review_text",
    "sentiment",
    "source"
}

assert expected_columns.issubset(df.columns), (
    f"Dataset columns are invalid. Found: {df.columns.tolist()}"
)

assert set(df["sentiment"].unique()) == {
    "positive",
    "negative"
}, "Expected exactly positive and negative labels."

print("✅ Input dataset is valid.")

## Expected Interpretation

หาก dataset ถูกสร้างแบบ balanced 400 positive และ 400 negative:

- Dataset ควรมี 800 แถว
- มี 4 columns ได้แก่ review_id, review_text, sentiment, source
- จำนวน positive และ negative ควรเท่ากัน
- จึงไม่เกิด class imbalance ในแลปหลัก
- อย่างไรก็ตาม ข้อความจริงยังมี noise เช่น emoji, URL, hashtag,
  การพิมพ์ซ้ำ และข้อความไทยปนอังกฤษ
- ขั้นตอน text cleaning จึงยังจำเป็น

In [2]:
print("Shape:", df.shape)

display(df.isnull().sum())

print("Duplicated rows:", df.duplicated().sum())

display(df["sentiment"].value_counts())

NameError: name 'df' is not defined

In [ ]:
#Text Cleaning
df = df.dropna(subset=["review_text", "sentiment"]).copy()

df = df.drop_duplicates(subset=["review_text"]).copy()

df["review_text"] = df["review_text"].astype(str)

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

def clean_text(text):
    text = str(text).lower()

    text = BeautifulSoup(text, "html.parser").get_text(" ")

    text = re.sub(r"http\S+|www\.\S+", " ", text)

    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(r"[^a-zA-Zก-๙\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["review_text"].apply(clean_text)

display(
    df[
        [
            "review_text",
            "clean_text",
            "sentiment"
        ]
    ].head(10)
)

# ANSWER

## Expected Interpretation: Text Cleaning

- URL และ HTML tags มักไม่ช่วยแยก sentiment โดยตรง
- การลดช่องว่างและ normalization ช่วยให้ vocabulary มีรูปแบบสม่ำเสมอ
- การลบ punctuation ลด noise แต่มีความเสี่ยงทำให้ emoji เช่น 😡 หรือ !!!
  ซึ่งอาจสื่ออารมณ์ หายไป
- คำปฏิเสธ เช่น “ไม่”, “ไม่ได้”, “not”, “never” ไม่ควรถูกลบเป็น stopwords
  เพราะสามารถเปลี่ยน polarity ของข้อความ เช่น “ดี” กับ “ไม่ดี”
- สำหรับภาษาไทย การตัดคำเป็นประเด็นสำคัญ แต่แลปนี้มุ่งสอน TF-IDF workflow
  จึงใช้ character/whitespace-based preprocessing แบบง่ายก่อน

In [ ]:
#Train/Test Split
X = df["clean_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nTrain label distribution:")
display(y_train.value_counts(normalize=True).round(3))

print("\nTest label distribution:")
display(y_test.value_counts(normalize=True).round(3))

## Expected Interpretation: Train/Test Split

- ใช้ training set เพื่อให้โมเดลเรียนรู้ความสัมพันธ์ระหว่างข้อความและ sentiment
- ใช้ test set เพื่อประเมินกับข้อมูลที่โมเดลไม่เคยเห็น
- `stratify=y` รักษาสัดส่วนของ positive และ negative ในทั้งสองชุด
- ไม่ควร fit TF-IDF บนข้อมูลทั้งหมดก่อน split เพราะค่า IDF และ vocabulary
  อาจรับรู้ข้อมูลจาก test set ทำให้เกิด data leakage
- Pipeline ช่วยให้ TF-IDF ถูก fit บน X_train เท่านั้นเมื่อเรียก `model.fit(X_train, y_train)`

In [ ]:
#Experiment A and B

#Function

def train_and_evaluate(
    X_train,
    X_test,
    y_train,
    y_test,
    ngram_range=(1, 1),
    min_df=2
):
    pipeline = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=ngram_range,
                min_df=min_df,
                max_df=0.90
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division=0
    )

    return {
        "model": pipeline,
        "y_pred": predictions,
        "accuracy": accuracy_score(y_test, predictions),
        "f1_negative": report["negative"]["f1-score"],
        "f1_positive": report["positive"]["f1-score"],
        "vocabulary_size": len(
            pipeline.named_steps["tfidf"].get_feature_names_out()
        )
    }

In [ ]:
#Experiment A
unigram_result = train_and_evaluate(
    X_train,
    X_test,
    y_train,
    y_test,
    ngram_range=(1, 1),
    min_df=2
)

In [ ]:
#Experiment B
bigram_result = train_and_evaluate(
    X_train,
    X_test,
    y_train,
    y_test,
    ngram_range=(1, 2),
    min_df=2
)

In [ ]:
#Comparison Table
comparison_df = pd.DataFrame([
    {
        "configuration": "Unigram (1,1)",
        "vocabulary_size": unigram_result["vocabulary_size"],
        "accuracy": unigram_result["accuracy"],
        "f1_negative": unigram_result["f1_negative"],
        "f1_positive": unigram_result["f1_positive"]
    },
    {
        "configuration": "Unigram + Bigram (1,2)",
        "vocabulary_size": bigram_result["vocabulary_size"],
        "accuracy": bigram_result["accuracy"],
        "f1_negative": bigram_result["f1_negative"],
        "f1_positive": bigram_result["f1_positive"]
    }
])

display(
    comparison_df.style.format({
        "accuracy": "{:.4f}",
        "f1_negative": "{:.4f}",
        "f1_positive": "{:.4f}"
    })
)

# Note
โมเดลที่เหมาะสมควรพิจารณา F1-score ของ negative class ร่วมกับ Accuracy
ไม่ควรเลือกจาก Accuracy เพียงอย่างเดียว

Bigram มักทำให้ vocabulary เพิ่มขึ้น และอาจช่วยให้โมเดลจับวลีร่วม
เช่น “ไม่ ดี” หรือ “ส่ง ช้า” ได้ดีขึ้น แต่ไม่ได้รับประกันว่าจะให้ผลดีที่สุดเสมอไป

In [ ]:
#Automatic Evaluation and model selection
best_result = max(
    [unigram_result, bigram_result],
    key=lambda result: result["f1_negative"]
)

best_model = best_result["model"]
y_pred_best = best_result["y_pred"]

print("Best model by F1 Negative:")
print("Accuracy:", round(best_result["accuracy"], 4))
print("F1 Negative:", round(best_result["f1_negative"], 4))
print("F1 Positive:", round(best_result["f1_positive"], 4))

In [ ]:
#Classification Report
print(
    classification_report(
        y_test,
        y_pred_best,
        labels=["negative", "positive"],
        zero_division=0
    )
)

In [ ]:
#Confusion Matrix
labels = ["negative", "positive"]

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best,
    labels=labels,
    display_labels=labels,
    cmap="Blues",
    values_format="d"
)

plt.title("Confusion Matrix: Best Model")
plt.grid(False)
plt.show()

# Note
How to Explanation
| Metric             | ความหมายในโจทย์นี้                                                          |
| ------------------ | --------------------------------------------------------------------------- |
| Accuracy           | สัดส่วนทั้งหมดที่ทำนายถูก                                                   |
| Precision Negative | เมื่อโมเดลทายว่า negative แล้ว ถูกจริงบ่อยเพียงใด                           |
| Recall Negative    | รีวิว negative จริงถูกตรวจพบโดยโมเดลมากเพียงใด                              |
| F1 Negative        | ค่ากลางที่สมดุลระหว่าง Precision และ Recall ของ negative                    |
| False Negative     | รีวิว negative ที่โมเดลทายเป็น positive — อาจทำให้ทีมพลาดลูกค้าที่ไม่พอใจ   |
| False Positive     | รีวิว positive ที่โมเดลทายเป็น negative — เพิ่มภาระให้ทีม support ตรวจเพิ่ม |

In [ ]:
#Error analysis
error_df = pd.DataFrame({
    "review_text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred_best
})

errors = error_df[
    error_df["actual"] != error_df["predicted"]
].copy()

print("Incorrect predictions:", len(errors))

display(errors.head(15))

## Expected Error Analysis

สาเหตุที่โมเดลอาจทำนายผิด:

1. ข้อความสั้นมาก เช่น “โอเค”, “ธรรมดา”, “แย่”
   ทำให้มี feature ไม่เพียงพอ

2. มี sentiment ผสม เช่น “ของดี แต่ส่งช้ามาก”
   ซึ่งอาจมีทั้งคำบวกและลบในข้อความเดียวกัน

3. มีการประชด เช่น “ดีมากเลย รอของเป็นเดือน”
   ซึ่ง TF-IDF + Logistic Regression เข้าใจได้ยาก

4. มีคำปฏิเสธหรือวลีซับซ้อน เช่น “ไม่ได้แย่อย่างที่คิด”
   ซึ่งลำดับและบริบทคำมีผลต่อความหมาย

5. Label ของข้อมูล social media อาจมีความคลุมเครือ
   หรือ sentiment ขึ้นกับบริบทที่ไม่มีอยู่ในข้อความ

In [ ]:
#Top Predictive Term
tfidf = best_model.named_steps["tfidf"]
classifier = best_model.named_steps["classifier"]

feature_names = tfidf.get_feature_names_out()
coefficients = classifier.coef_[0]
classes = classifier.classes_

print("Classes:", classes)

positive_index = list(classes).index("positive")

if positive_index == 1:
    top_positive_indices = np.argsort(coefficients)[-15:][::-1]
    top_negative_indices = np.argsort(coefficients)[:15]
else:
    top_positive_indices = np.argsort(coefficients)[:15]
    top_negative_indices = np.argsort(coefficients)[-15:][::-1]

top_positive_terms = feature_names[top_positive_indices]
top_negative_terms = feature_names[top_negative_indices]

print("Top Positive Terms:")
print(top_positive_terms)

print("\nTop Negative Terms:")
print(top_negative_terms)

## Sample Business Insight

โมเดล TF-IDF ร่วมกับ Logistic Regression สามารถใช้เป็น baseline
เพื่อคัดกรอง customer feedback เชิงลบแบบอัตโนมัติได้

ในการใช้งานจริง ควรให้ทีม customer support ตรวจสอบข้อความที่โมเดลทำนายว่า
negative ก่อน เพื่อจัดลำดับการตอบกลับและลดโอกาสพลาดลูกค้าที่ไม่พอใจ

หากเป้าหมายหลักคือการค้นหารีวิวเชิงลบให้ได้มากที่สุด ควรพิจารณา
Recall ของ negative class เป็นหลัก และอาจยอมให้เกิด false positive เพิ่มขึ้นได้

ข้อจำกัดสำคัญคือ TF-IDF ไม่เข้าใจบริบท ลำดับคำ การประชด และคำปฏิเสธที่ซับซ้อน
จึงควรเก็บข้อมูลเพิ่ม ตรวจ label และทดลอง Thai tokenizer, character n-gram
หรือ Transformer model ในงานขั้นต่อไป

# Quick Grading Rubric

| หัวข้อ | หลักฐานขั้นต่ำที่ต้องมี | คะแนน |
|---|---|---:|
| Data Exploration | shape, missing, duplicate, class distribution | 10 |
| Text Cleaning | สร้าง clean_text และอธิบายผล cleaning | 20 |
| Train/Test Split | ใช้ test_size=0.20 และ stratify=y | 10 |
| Model Experiments | ทดลอง unigram และ bigram หรือเหตุผลการเลือก | 20 |
| Evaluation | classification report และ confusion matrix | 20 |
| Error Analysis | วิเคราะห์ข้อความทำนายผิดอย่างน้อย 3 รายการ | 10 |
| Business Insight | เชื่อม metrics กับงาน customer support | 10 |
| **รวม** |  | **100** |

In [ ]:
#Save model
import joblib

MODEL_FILE = "sentiment_tfidf_logistic_regression.joblib"

joblib.dump(
    best_model,
    MODEL_FILE
)

print(f"Saved: {MODEL_FILE}")

In [ ]:
#Load model
loaded_model = joblib.load(MODEL_FILE)

new_messages = [
    "บริการดีมาก ส่งเร็วและคุ้มราคา",
    "แย่มาก ของเสียและติดต่อไม่ได้",
    "สินค้าดีแต่จัดส่งช้าไปหน่อย"
]

predictions = loaded_model.predict(new_messages)

prediction_df = pd.DataFrame({
    "text": new_messages,
    "prediction": predictions
})

display(prediction_df)